In [ ]:
import json
import networkx as nx
from pyvis.network import Network

# =====================================================
# 1. 构建图
# =====================================================
def build_graph(all_results):
    G = nx.DiGraph()
    # 兼容两种格式：如果是字典(按文件名分类)，或是直接的三元组列表
    if isinstance(all_results, dict):
        for _, triples in all_results.items():
            for t in triples:
                G.add_edge(t["cause"], t["effect"], label=t["relationship_type"])
    elif isinstance(all_results, list):
        for t in all_results:
            G.add_edge(t["cause"], t["effect"], label=t["relationship_type"])
    return G

# =====================================================
# 2. 识别“最直接后果链”（贪心算法）
# =====================================================
def identify_main_risk_chain(G):
    if len(G.nodes()) == 0:
        return []
    
    # 找到影响力核心点
    scores = {node: G.out_degree(node) + G.in_degree(node) for node in G.nodes()}
    start_node = max(scores, key=scores.get)
    
    main_chain = [start_node]
    current_node = start_node
    visited = {start_node}

    while True:
        successors = list(G.successors(current_node))
        if not successors:
            break
        # 贪心选择：走后续影响力（出度）最大的支路
        next_node = max(successors, key=lambda n: G.out_degree(n))
        if next_node in visited:
            break
        main_chain.append(next_node)
        visited.add(next_node)
        current_node = next_node
    return main_chain

# =====================================================
# 3. 计算最长路径
# =====================================================
def compute_longest_path(G):
    # 如果有环，进行传递约简处理以计算路径
    if not nx.is_directed_acyclic_graph(G):
        G_dag = nx.DiGraph(nx.algorithms.dag.transitive_reduction(G))
    else:
        G_dag = G
    try:
        return nx.dag_longest_path(G_dag)
    except:
        return []

# =====================================================
# 4. 可视化函数
# =====================================================
def visualize_path(G, path_nodes, filename):
    if not path_nodes:
        print(f"⚠️ 路径为空，跳过 {filename}")
        return

    net = Network(height="800px", width="100%", directed=True, notebook=False)
    net.barnes_hut()

    path_nodes_set = set(path_nodes)
    # 构造路径中的边集，用于高亮
    path_edges = set(zip(path_nodes, path_nodes[1:]))

    for node in G.nodes():
        color = "#ff4d4d" if node in path_nodes_set else "#97c2fc"
        net.add_node(node, label=str(node), color=color, title=f"节点: {node}")

    for source, target, data in G.edges(data=True):
        is_in_path = (source, target) in path_edges
        color = "#ff4d4d" if is_in_path else "#cccccc"
        width = 3 if is_in_path else 1
        net.add_edge(source, target, label=data.get("label", ""), color=color, width=width, arrows="to")

    net.save_graph(filename)
    print(f"✅ 可视化报告已生成: {filename}")

# =====================================================
# 执行入口
# =====================================================
def main():
    input_file = "all_triples.json"
    
    try:
        with open(input_file, "r", encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"❌ 错误：找不到文件 {input_file}，请先运行抽取程序。")
        return

    # 构建并分析
    G = build_graph(data)
    print(f"📊 图谱构建完成：{G.number_of_nodes()} 节点, {G.number_of_edges()} 条边")

    # 1. 识别主后果链
    main_chain = identify_main_risk_chain(G)
    visualize_path(G, main_chain, "report_main_chain.html")

    # 2. 识别最长演化路径
    longest_path = compute_longest_path(G)
    visualize_path(G, longest_path, "report_longest_path.html")

if __name__ == "__main__":
    main()